# Notebook 06: OCR Model Evaluation, Inference & CER/WER Benchmark
**Phase:** 05 Document Processing and OCR  
**Thesis:** A Multimodal Machine Learning Approach Integrating CNN and Transformer Architectures for Charity Patient Eligibility Classification  
**Institution:** Ophthalmological Foundation of the Philippines (OFPHIL)  

---
### 1. Isolated Architecture: No File Overlap
To completely prevent file clobbering and maintain experimental integrity, the pipeline is separated into two dedicated directories:
- **Directory 1 (`standardized/`):** Where you manually annotate, label, and transcribe in the Annotator App. (Golden Ground Truth — strictly protected from model writes).
- **Directory 2 (`model_predictions_and_benchmarks/`):** Where this notebook executes OCR inference, computes CER & WER, and saves AI predictions and evaluation reports without overwriting your original ground truth!

**Workflow:**
1. Reads Ground Truth from `data/interim/document_processing/standardized/`.
2. Crops bounding boxes from original high-resolution document images.
3. Feeds image crops to **PaddleOCR PP-OCRv6** to generate **`ocr_predicted_text`**.
4. Calculates mathematical Character Error Rate (**CER**) and Word Error Rate (**WER**).
5. Exports results exclusively to **`data/interim/document_processing/model_predictions_and_benchmarks/`**.

In [2]:
# Step 1: Directory Setup & Input Verification
import os
import sys
import json
import cv2
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

# I-suppress ang unnecessary warnings
warnings.filterwarnings("ignore")

# 1. Project Root Directory
PROJECT_ROOT = Path(r"c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\dss-charity-eligibity-classification")
BASE_DIR = PROJECT_ROOT / "data" / "interim" / "document_processing"

# 2. Input Directory (Standardized Golden Ground Truth)
GT_DIR = BASE_DIR / "standardized"
GT_DET_CSV = GT_DIR / "text_detection_annotations.csv"
GT_OCR_CSV = GT_DIR / "ocr_transcription_annotations.csv"

# 3. Output Directory (Model predictions & benchmarks)
PRED_DIR = BASE_DIR / "model_predictions_and_benchmarks"
PRED_DIR.mkdir(parents=True, exist_ok=True)
PRED_DET_CSV = PRED_DIR / "model_detection_predictions.csv"
PRED_OCR_CSV = PRED_DIR / "model_ocr_transcription_benchmark.csv"

print("=== Directory Architecture ===")
print("Project Root                 :", PROJECT_ROOT)
print("Ground Truth Dir (Read-Only) :", GT_DIR)
print("Model Benchmark Dir (Output) :", PRED_DIR)
print("----------------------------------------------------")
print("Ground Truth Detection CSV     :", GT_DET_CSV.exists())
print("Ground Truth Transcription CSV :", GT_OCR_CSV.exists())

# 4. Load DataFrames into Memory
df_det = pd.read_csv(GT_DET_CSV)
df_ocr = pd.read_csv(GT_OCR_CSV)

print(f"Total Ground Truth Detection Boxes : {len(df_det)}")
print(f"Total Ground Truth Transcriptions  : {len(df_ocr)}")
df_ocr.head()


=== Directory Architecture ===
Project Root                 : c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\dss-charity-eligibity-classification
Ground Truth Dir (Read-Only) : c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\dss-charity-eligibity-classification\data\interim\document_processing\standardized
Model Benchmark Dir (Output) : c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\dss-charity-eligibity-classification\data\interim\document_processing\model_predictions_and_benchmarks
----------------------------------------------------
Ground Truth Detection CSV     : True
Ground Truth Transcription CSV : True
Total Ground Truth Detection Boxes : 18288
Total Ground Truth Transcriptions  : 18288


,transcription_id,file_name,relative_path,document_category,text_modality,ocr_predicted_text,ground_truth_transcription,character_error_rate_cer,word_error_rate_wer,is_fully_transcribed,transcription_quality_notes,box_id,label_name,box_type,ocr_model_confidence,is_ground_truth_verified,annotation_status,transcription_confidence
0,1,abs_p_001.jpg,data/raw/supporting document/supporting_docu/p...,assessment,handwritten,NaN,nephroplus,NaN,NaN,True,Manual human ground truth,1.0,hospital_name,FORM_SECTION,NaN,True,verified,1.0
1,2,abs_p_001.jpg,data/raw/supporting document/supporting_docu/p...,assessment,handwritten,NaN,NEPHROCARE HEALTH CARE SERVICES PHILIPPINES INC.,NaN,NaN,True,Manual human ground truth,2.0,hospital_name,FORM_SECTION,NaN,True,verified,1.0
2,3,abs_p_001.jpg,data/raw/supporting document/supporting_docu/p...,assessment,handwritten,NaN,182 VBM Building A. Bonifacio Street Bgy. Taño...,NaN,NaN,True,Manual human ground truth,3.0,address,FORM_SECTION,NaN,True,verified,1.0
3,4,abs_p_001.jpg,data/raw/supporting document/supporting_docu/p...,assessment,handwritten,NaN,09694283305,NaN,NaN,True,Manual human ground truth,4.0,contact_no,FORM_SECTION,NaN,True,verified,1.0
4,5,abs_p_001.jpg,data/raw/supporting document/supporting_docu/p...,assessment,handwritten,NaN,ph.mrk.abs.cm@nephroplus.com,NaN,NaN,True,Manual human ground truth,5.0,contact_no,FORM_SECTION,NaN,True,verified,1.0


In [3]:
# Step 2: Define Mathematical CER & WER Distance Calculators
def levenshtein_distance(s1, s2):
    """Calculates Levenshtein edit distance between two sequences."""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]

def compute_cer(reference, hypothesis):
    """Calculates Character Error Rate (CER)."""
    ref = str(reference).strip()
    hyp = str(hypothesis).strip()
    if not ref:
        return 0.0 if not hyp else 1.0
    dist = levenshtein_distance(ref, hyp)
    return round(min(1.0, dist / len(ref)), 4)

def compute_wer(reference, hypothesis):
    """Calculates Word Error Rate (WER)."""
    ref_words = str(reference).strip().split()
    hyp_words = str(hypothesis).strip().split()
    if not ref_words:
        return 0.0 if not hyp_words else 1.0
    dist = levenshtein_distance(ref_words, hyp_words)
    return round(min(1.0, dist / len(ref_words)), 4)

print('Metrics Sanity Check (Expected: 0.0): CER =', compute_cer('ANDRALES', 'ANDRALES'), '| WER =', compute_wer('ANDRALES', 'ANDRALES'))
print('Metrics Sanity Check (Expected > 0): CER =', compute_cer('0626-02', '624-0'), '| WER =', compute_wer('0626-02', '624-0'))


Metrics Sanity Check (Expected: 0.0): CER = 0.0 | WER = 0.0
Metrics Sanity Check (Expected > 0): CER = 0.4286 | WER = 1.0


In [4]:
# Step 3: Initialize PaddleOCR Recognition Pipeline (Silent Mode - Zero Console Spam)
import os
import sys
import io
import contextlib
import warnings

# Suppress internal Paddle C++ logs and Python warnings
warnings.filterwarnings("ignore")
os.environ["GLOG_minloglevel"] = "3"
os.environ["PADDLE_PDX_LOGGING_LEVEL"] = "ERROR"

try:
    # I-redirect ang stdout at stderr habang naglo-load ang model para walang lalabas na spam
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        from paddleocr import PaddleOCR
        ocr_engine = PaddleOCR(use_textline_orientation=True, lang='en')
        
    print('[SUCCESS] PaddleOCR initialized successfully with PP-OCRv6 models.')
except Exception as e:
    print(f'[ERROR] PaddleOCR initialization failed ({e}).')
    ocr_engine = None

[SUCCESS] PaddleOCR initialized successfully with PP-OCRv6 models.


In [5]:
# Step 4: Run OCR Inference on Cropped Image Boxes & Compute CER/WER (Using predict & clean filtering)
import warnings
warnings.filterwarnings("ignore")

evaluated_records = []
det_conf_map = {}

# Group by document (file_name) so each image is loaded once, processed, and immediately freed from RAM
doc_groups = df_ocr.groupby('file_name', sort=False)
total_docs = len(doc_groups)
print(f"Starting inference across {total_docs} documents ({len(df_ocr)} boxes total)...")

doc_idx = 0
for file_name, group in doc_groups:
    doc_idx += 1
    rel_path = group['relative_path'].iloc[0] if 'relative_path' in group.columns else None
    img = None
    
    if rel_path:
        img_path = PROJECT_ROOT / rel_path
        if img_path.exists():
            img = cv2.imread(str(img_path))
            
    for _, row in group.iterrows():
        box_id = row['box_id']
        gt_text = str(row.get('ground_truth_transcription', '')).strip()
        box_type = str(row.get('box_type', ''))
        
        # Flag if this is a non-text symbol (e.g. signature or pure checkbox)
        is_pure_symbol = box_type in ['SIGNATURE', 'TABLE_CELL', 'CHECKMARK_OR_CHECKBOX'] or gt_text == 'nan' or not gt_text
        
        det_match = df_det[(df_det['file_name'] == file_name) & (df_det['box_id'] == box_id)]
        predicted_text = ""
        pred_conf = 0.0
        
        # Only run text OCR prediction if it's an actual legible text entry with valid bounding box
        if not is_pure_symbol and img is not None and not det_match.empty:
            box_info = det_match.iloc[0]
            bx = int(box_info.get('bbox_x', 0))
            by = int(box_info.get('bbox_y', 0))
            bw = int(box_info.get('bbox_width', 0))
            bh = int(box_info.get('bbox_height', 0))
            
            if bw > 5 and bh > 5:
                crop = img[by:by+bh, bx:bx+bw]
                if crop.size > 0 and ocr_engine is not None:
                    try:
                        # Gamitin ang predict() sa halip na ocr() para iwasan ang deprecation warning
                        res = ocr_engine.predict(crop)
                        if res and len(res) > 0 and res[0] is not None:
                            item = res[0]
                            # Handle OCRResult object / dict mula sa PP-OCRv6 predict()
                            if hasattr(item, 'get') or isinstance(item, dict):
                                texts = item.get('rec_texts', [])
                                scores = item.get('rec_scores', [])
                                predicted_text = ' '.join(texts).strip()
                                pred_conf = float(np.mean(scores)) if scores else 0.0
                            elif hasattr(item, 'rec_texts'):
                                texts = getattr(item, 'rec_texts', [])
                                scores = getattr(item, 'rec_scores', [])
                                predicted_text = ' '.join(texts).strip()
                                pred_conf = float(np.mean(scores)) if scores else 0.0
                            # Legacy format fallback [[box, (text, score)]]
                            elif isinstance(item, list):
                                texts = [line[1][0] for line in item if len(line) > 1 and len(line[1]) > 0]
                                confs = [line[1][1] for line in item if len(line) > 1 and len(line[1]) > 1]
                                predicted_text = ' '.join(texts).strip()
                                pred_conf = float(np.mean(confs)) if confs else 0.0
                    except Exception:
                        pass
                        
        # Compute actual CER and WER
        if is_pure_symbol:
            cer = np.nan
            wer = np.nan
        else:
            cer = compute_cer(gt_text, predicted_text)
            wer = compute_wer(gt_text, predicted_text)
        
        rec = dict(row)
        rec['ocr_predicted_text'] = predicted_text
        rec['character_error_rate_cer'] = cer
        rec['word_error_rate_wer'] = wer
        rec['ocr_model_confidence'] = round(pred_conf, 4) if pred_conf > 0 else 0.0
        evaluated_records.append(rec)
        det_conf_map[(file_name, box_id)] = round(pred_conf, 4) if pred_conf > 0 else 0.0
        
    # Free image memory immediately to prevent RAM crash
    del img
    
    # Live Progress Output every 50 documents
    if doc_idx % 50 == 0 or doc_idx == total_docs:
        print(f"Processed [{doc_idx}/{total_docs}] documents ({len(evaluated_records)} boxes evaluated)...")

df_evaluated = pd.DataFrame(evaluated_records)
print(f"\n[SUCCESS] Successfully evaluated all {len(df_evaluated)} text boxes without memory error!")

# Ipakita ang malinis na summary para sa legible text
df_text_only = df_evaluated[df_evaluated['character_error_rate_cer'].notnull()]
print(f"Valid Pure-Text Crops Evaluated: {len(df_text_only)}")
print(f"Overall Mean CER: {df_text_only['character_error_rate_cer'].mean() * 100:.2f}%")
print(f"Overall Mean WER: {df_text_only['word_error_rate_wer'].mean() * 100:.2f}%")

df_evaluated[['transcription_id', 'label_name', 'box_type', 'ground_truth_transcription', 'ocr_predicted_text', 'character_error_rate_cer', 'word_error_rate_wer', 'ocr_model_confidence']].head(10)


Starting inference across 1017 documents (18288 boxes total)...
Processed [50/1017] documents (2521 boxes evaluated)...
Processed [100/1017] documents (3881 boxes evaluated)...
Processed [150/1017] documents (4421 boxes evaluated)...
Processed [200/1017] documents (4950 boxes evaluated)...
Processed [250/1017] documents (5634 boxes evaluated)...
Processed [300/1017] documents (6620 boxes evaluated)...
Processed [350/1017] documents (7396 boxes evaluated)...
Processed [400/1017] documents (8054 boxes evaluated)...
Processed [450/1017] documents (8694 boxes evaluated)...
Processed [500/1017] documents (9320 boxes evaluated)...
Processed [550/1017] documents (10821 boxes evaluated)...
Processed [600/1017] documents (11730 boxes evaluated)...
Processed [650/1017] documents (12320 boxes evaluated)...
Processed [700/1017] documents (13338 boxes evaluated)...
Processed [750/1017] documents (14199 boxes evaluated)...
Processed [800/1017] documents (15105 boxes evaluated)...
Processed [850/1017

,transcription_id,label_name,box_type,ground_truth_transcription,ocr_predicted_text,character_error_rate_cer,word_error_rate_wer,ocr_model_confidence
0,1,hospital_name,FORM_SECTION,nephroplus,Inephroplu,0.2000,1.0000,0.9195
1,2,hospital_name,FORM_SECTION,NEPHROCARE HEALTH CARE SERVICES PHILIPPINES INC.,PHROCAREHEALTHCARESERVICESPHILIPPIN,0.2708,1.0000,0.9741
2,3,address,FORM_SECTION,182 VBM Building A. Bonifacio Street Bgy. Taño...,182 VBM Building A. Bonifacio Street Bgy. Tari...,0.1111,0.3000,0.9691
3,4,contact_no,FORM_SECTION,09694283305,t: 09694283305,0.2727,1.0000,0.9879
4,5,contact_no,FORM_SECTION,ph.mrk.abs.cm@nephroplus.com,mix.abs.cml@nephroplus.com,0.2143,1.0000,0.9419
5,6,document_title,FORM_SECTION,MEDICAL ABSTRACT,ICAL ABSTRACT,0.1875,0.5000,0.9965
6,7,date_issue,HANDWRITTEN_ENTRY,"July 15, 2025","July 15, 2025",0.0000,0.0000,0.9861
7,8,patient_name,HANDWRITTEN_ENTRY,"EVANGELISTA, EDGAR SUMALJAG","EVANGELISTA, EDGAR SUMALJA",0.0370,0.3333,0.9899
8,9,date_of_birth,HANDWRITTEN_ENTRY,August 27 1963,"August 27,1963",0.0714,0.6667,0.9685
9,10,diagnosis,HANDWRITTEN_ENTRY,Chronic Kidney Disease Stage 5 secondary to Di...,sasc Ksioney Disease Stage 5 secondary to Diab...,0.3137,0.4667,0.8514


In [6]:
# Step 5: Export Predictions to model_predictions_and_benchmarks/ (Zero Overlap with Ground Truth!)

# 1. Save Detection Predictions with Model Confidence to model_predictions_and_benchmarks/
df_det_pred = df_det.copy()
for idx, row in df_det_pred.iterrows():
    key = (row['file_name'], row['box_id'])
    if key in det_conf_map:
        df_det_pred.at[idx, 'confidence_score'] = det_conf_map[key]

df_det_pred.to_csv(PRED_DET_CSV, index=False)
print(f'[SUCCESS] Saved Detection Predictions to: {PRED_DET_CSV.relative_to(PROJECT_ROOT)}')

# 2. Save OCR Benchmark with Predicted Text, CER & WER to model_predictions_and_benchmarks/
df_evaluated.to_csv(PRED_OCR_CSV, index=False)
print(f'[SUCCESS] Saved OCR Benchmark to: {PRED_OCR_CSV.relative_to(PROJECT_ROOT)}')

# 3. Filter Valid Pure-Text Entries for Legitimate OCR Scoring
# Non-text symbols (signatures, pure checkmarks, unreadable) are excluded from the CER/WER denominator
df_text_only = df_evaluated[df_evaluated['character_error_rate_cer'].notnull()].copy()

mean_cer = df_text_only['character_error_rate_cer'].mean()
mean_wer = df_text_only['word_error_rate_wer'].mean()
accuracy_char = round((1.0 - mean_cer) * 100, 2)
accuracy_word = round((1.0 - mean_wer) * 100, 2)

print('\n====================================================')
print('      OFPHIL PADDLEOCR PURE-TEXT BENCHMARK REPORT    ')
print('====================================================')
print(f'Total Crops Evaluated           : {len(df_evaluated)}')
print(f'Valid Legible Text Crops        : {len(df_text_only)}')
print(f'Excluded Non-Text / Symbols     : {len(df_evaluated) - len(df_text_only)}')
print('----------------------------------------------------')
print(f'Mean Character Error Rate (CER) : {mean_cer:.4f} (Accuracy: {accuracy_char:.2f}%)')
print(f'Mean Word Error Rate (WER)      : {mean_wer:.4f} (Accuracy: {accuracy_word:.2f}%)')
print('----------------------------------------------------')

# Breakdown by text_modality (handwritten vs printed)
modality_bench = df_text_only.groupby('text_modality')[['character_error_rate_cer', 'word_error_rate_wer', 'ocr_model_confidence']].mean()
print('\nBreakdown by Modality (Handwritten vs Printed):')
print(modality_bench)

# Breakdown by box_type
box_type_bench = df_text_only.groupby('box_type')[['character_error_rate_cer', 'word_error_rate_wer', 'ocr_model_confidence']].mean()
print('\nBreakdown by Box Type:')
print(box_type_bench)

# 4. Save JSON Summary Report in model_predictions_and_benchmarks/
summary_report = {
    'model_name': 'PaddleOCR (PP-OCRv6)',
    'total_boxes_evaluated': len(df_evaluated),
    'valid_text_crops_evaluated': len(df_text_only),
    'excluded_non_text_count': len(df_evaluated) - len(df_text_only),
    'overall_mean_cer': round(mean_cer, 4),
    'overall_mean_wer': round(mean_wer, 4),
    'overall_char_accuracy_pct': accuracy_char,
    'overall_word_accuracy_pct': accuracy_word,
    'by_modality': modality_bench.to_dict(orient='index'),
    'by_box_type': box_type_bench.to_dict(orient='index')
}

summary_path = PRED_DIR / 'paddleocr' / 'paddleocr_evaluation_benchmark_report.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary_report, f, indent=4)
print(f'\n[SUCCESS] Benchmark report exported to: {summary_path.relative_to(PROJECT_ROOT)}')
print('[COMPLETE] Ground Truth files in standardized/ are 100% untouched and preserved!')


[SUCCESS] Saved Detection Predictions to: data\interim\document_processing\model_predictions_and_benchmarks\model_detection_predictions.csv
[SUCCESS] Saved OCR Benchmark to: data\interim\document_processing\model_predictions_and_benchmarks\model_ocr_transcription_benchmark.csv

      OFPHIL PADDLEOCR PURE-TEXT BENCHMARK REPORT    
Total Crops Evaluated           : 18288
Valid Legible Text Crops        : 14614
Excluded Non-Text / Symbols     : 3674
----------------------------------------------------
Mean Character Error Rate (CER) : 0.5316 (Accuracy: 46.84%)
Mean Word Error Rate (WER)      : 0.7754 (Accuracy: 22.46%)
----------------------------------------------------

Breakdown by Modality (Handwritten vs Printed):
               character_error_rate_cer  word_error_rate_wer  \
text_modality                                                  
handwritten                    0.622249             0.862135   
mixed                          0.566078             0.849031   
printed          

### 6. Conclusions & Thesis Recommendations
- **Ground Truth Integrity:** Ground Truth files remain strictly isolated in `standardized/`.
- **Prediction Transparency:** All AI inferences, confidence ratings, and error calculations are saved under `model_predictions_and_benchmarks/`.
- **Model Gap Analysis:** The Character Error Rate (CER) and Word Error Rate (WER) benchmark accurately demonstrates challenges with raw medical handwriting, supporting the thesis rationale for multimodal classification in Phase 06.